# Scaling study — Qwen2.5 7B / 14B on a free Colab T4

**What this is for.** The CPU-only 3B results are the project's primary contribution. This notebook adds a **scaling curve** on top of them: 3B → 7B → 14B, with everything except model scale held constant.

The point is not a bigger accuracy number. It is the **shape of the curve** relative to the clinical safety bars (low 70% / medium 80% / high 90%). A single model's ceiling is an observation; a curve is an *extrapolation* — it says whether the bars are reachable by scaling **at all**.

**Runtime — Colab free tier caps sessions at ~12 h and disconnects when idle:**

| Model | Q4_K_M size | Fits T4 (16 GB)? | All policies |
|---|---|---|---|
| Qwen2.5-7B | ~4.5 GB | yes | 11–18 h (~2 sessions) |
| Qwen2.5-14B | ~9 GB | yes, tight | 22–35 h (~3+ sessions) |
| Qwen2.5-32B | ~19 GB | **no** — exceeds VRAM | — |

**Everything here is resumable.** Logs live on Google Drive and `run_experiment.py` skips question-ids already present, so a disconnect costs minutes, not the run. Re-run the notebook top-to-bottom after any disconnect.

**Do not skip the smoke test (step 6).** Qwen uses ChatML (`<|im_start|>`), not Llama's `[INST]`. The wrong markup does not raise an error — it silently degrades the answers, and you would be measuring the prompt rather than the model. The smoke test is what stands between you and a wasted 22 hours.

## 1. Check the GPU

Runtime → Change runtime type → **T4 GPU**. If this cell shows no GPU, fix that before continuing — everything below assumes CUDA.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 2. Mount Drive

Drive is what makes this resumable: the model weights, the indexes and the logs all
persist there across sessions, so a disconnect never destroys work.

**One-time setup on your own machine.** Supply retrieval one of two ways:

**A — prebuilt indexes (preferred, 1.81 GB).** No on-box rebuild, and it skips the
assemble stage, which is the only step here that can run out of RAM. Upload:

```
MyDrive/medrag/indexes/bm25_medcorp_tp.pkl              (786 MB)
MyDrive/medrag/indexes/faiss_medcorp_tp/faiss.index     (624 MB)
MyDrive/medrag/indexes/faiss_medcorp_tp/chunks.pkl      (401 MB)
```

Do **not** upload `indexes/faiss_medcorp_tp/_shards/` (213 files, 639 MB) — those are
embed checkpoints used only when rebuilding. The corpus JSONL is not needed either:
`chunks.pkl` already carries the chunk text.

**B — corpus only (fallback, 412 MB).** Cheaper upload, but costs ~25 min of GPU
rebuild and carries the OOM risk:

```
MyDrive/medrag/data/corpora/medcorp_tp.jsonl
```

The notebook detects which you provided and takes the right path.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/medrag'
for sub in ('logs', 'models', 'indexes', 'data/corpora'):
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)

# Two ways to supply retrieval, checked in order of preference:
#
#   A. PREBUILT INDEXES on Drive (1.81 GB) -- preferred. No rebuild, and it skips
#      the assemble stage, which is the only step here that can OOM.
#   B. CORPUS JSONL on Drive (412 MB) -- fallback. Rebuilt on the GPU (~25 min).
#
# With prebuilt indexes the corpus is not needed at runtime: chunks.pkl already
# carries the chunk text.
corpus = f'{DRIVE}/data/corpora/medcorp_tp.jsonl'
HAVE_CORPUS = os.path.exists(corpus)
HAVE_INDEXES = (os.path.exists(f'{DRIVE}/indexes/bm25_medcorp_tp.pkl') and
                os.path.exists(f'{DRIVE}/indexes/faiss_medcorp_tp/faiss.index') and
                os.path.exists(f'{DRIVE}/indexes/faiss_medcorp_tp/chunks.pkl'))

print('prebuilt indexes on Drive:', HAVE_INDEXES)
print('corpus jsonl on Drive    :', HAVE_CORPUS,
      f'({round(os.path.getsize(corpus) / 1e6)} MB)' if HAVE_CORPUS else '')

assert HAVE_INDEXES or HAVE_CORPUS, (
    'Neither prebuilt indexes nor the corpus were found on Drive.\n'
    'Upload EITHER (preferred, 1.81 GB):\n'
    '  MyDrive/medrag/indexes/bm25_medcorp_tp.pkl\n'
    '  MyDrive/medrag/indexes/faiss_medcorp_tp/faiss.index\n'
    '  MyDrive/medrag/indexes/faiss_medcorp_tp/chunks.pkl\n'
    'OR (fallback, 412 MB):\n'
    '  MyDrive/medrag/data/corpora/medcorp_tp.jsonl'
)


## 3. Clone the repo and install

`llama-cpp-python` is built **with CUDA** here. That single flag is the whole GPU story for this project: llama-cpp under CUDA still exposes `_scores` (entropy gate) and `logprobs=2` (margin gate), so the gates work unchanged.

This is why the backend is not swapped for vLLM or TGI — those serve faster but do not expose per-token logits usefully, which would **break the gates** and with them the entire contribution being measured.

The build takes ~5 minutes.

In [ ]:
%cd /content
![ -d FinalProject_KCL ] || git clone https://github.com/Ideapersie/FinalProject_KCL.git
%cd /content/FinalProject_KCL

# CUDA build of llama-cpp-python (~5 min).
#
# PINNED to 0.3.28 — the version the gates are proven against locally. The entropy
# gate reads `_scores`, a semi-private llama-cpp attribute (see the note in
# llama_backend.py), and the margin gate relies on `logprobs=2`. A newer release
# that moves either one does not raise: the gates return None and P5 quietly
# degrades into retrieve-everything. Do not float this version.
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python==0.3.28 --force-reinstall --no-cache-dir
!pip install -q sentence-transformers faiss-cpu rank-bm25 pydantic pyyaml datasets huggingface_hub

import llama_cpp
print('llama-cpp-python', llama_cpp.__version__)


In [ ]:
# Confirm llama-cpp actually sees the GPU. If this says False, the CUDA build
# silently fell back to CPU and a 14B run would take days rather than hours.
from llama_cpp import llama_supports_gpu_offload
print('GPU offload available:', llama_supports_gpu_offload())
assert llama_supports_gpu_offload(), 'CUDA build failed — re-run the install cell.'

## 4. Download the model weights (Q4_K_M)

**Q4_K_M for both models, matching the 3B baseline's quantisation.** This matters: a higher-fidelity quant here would confound *bigger model* with *less quantisation*, and the curve would no longer isolate scale.

Weights are cached on Drive, so this only downloads once across all sessions.

In [ ]:
# Pick ONE per session. Start with 7B — it is faster and validates the pipeline
# before you commit a 22-hour 14B run to it.
MODEL = '7b'          # '7b' or '14b'

SPECS = {
    '7b':  dict(repo='Qwen/Qwen2.5-7B-Instruct-GGUF',
                file='qwen2.5-7b-instruct-q4_k_m.gguf',
                cfg='configs/models/qwen7b.yaml'),
    '14b': dict(repo='Qwen/Qwen2.5-14B-Instruct-GGUF',
                file='qwen2.5-14b-instruct-q4_k_m.gguf',
                cfg='configs/models/qwen14b.yaml'),
}
spec = SPECS[MODEL]

# Qwen's GGUF repos sometimes ship Q4_K_M as multiple shards
# (`...-q4_k_m-00001-of-00002.gguf`) rather than one file. Assuming the single-file
# name and being wrong costs a whole evening, so resolve it against the actual
# repo listing before downloading anything.
from huggingface_hub import list_repo_files
files = [f for f in list_repo_files(spec['repo']) if f.lower().endswith('.gguf')]
q4 = sorted(f for f in files if 'q4_k_m' in f.lower())
print('Q4_K_M files in repo:', q4)
assert q4, f'no Q4_K_M gguf in {spec["repo"]} — available: {files}'

if len(q4) > 1:
    # Sharded: download every shard; llama-cpp is given the FIRST one and finds
    # the rest by naming convention.
    print(f'sharded across {len(q4)} files')
    spec['file'] = q4[0]
    spec['shards'] = q4
else:
    spec['file'] = q4[0]
    spec['shards'] = q4

import os
os.makedirs('models', exist_ok=True)

from huggingface_hub import hf_hub_download
for shard in spec['shards']:
    cached = f"{DRIVE}/models/{os.path.basename(shard)}"
    if not os.path.exists(cached):
        print('downloading', shard)
        hf_hub_download(repo_id=spec['repo'], filename=shard,
                        local_dir=f'{DRIVE}/models', local_dir_use_symlinks=False)
    local = f"models/{os.path.basename(shard)}"
    if not os.path.exists(local):
        os.symlink(os.path.abspath(f"{DRIVE}/models/{os.path.basename(shard)}"), local)

GGUF = f"models/{os.path.basename(spec['file'])}"
print(f"{GGUF}: {os.path.getsize(GGUF) / 1e9:.1f} GB")

# The model YAML hard-codes the single-file name. If the repo turned out to be
# sharded, point the run at the real path via --model instead of editing the YAML.
GGUF_OVERRIDE = [] if GGUF == f"models/{SPECS[MODEL]['file']}" else ['--model', GGUF]
print('gguf override args:', GGUF_OVERRIDE or '(none needed)')


## 5. Get the indexes onto the box

If you uploaded the prebuilt indexes (option A), this just copies them to local disk
(~2 min) — local rather than the Drive mount, because FAISS and BM25 do heavy random
reads at query time and Drive FUSE is slow for that.

If you uploaded only the corpus (option B), this rebuilds on the GPU instead
(~25 min) and caches the result to Drive so later sessions skip it.

In [ ]:
import os, shutil, time

BM25 = 'indexes/bm25_medcorp_tp.pkl'
FAISS = 'indexes/faiss_medcorp_tp'
os.makedirs('indexes', exist_ok=True)
os.makedirs('data/corpora', exist_ok=True)

if HAVE_INDEXES:
    # Copy to local disk rather than reading over the Drive FUSE mount: FAISS and
    # BM25 do heavy random reads at query time and Drive is slow for that. ~2 min.
    if not os.path.exists(BM25):
        t = time.time()
        print('copying prebuilt indexes from Drive (~1.8 GB)...')
        shutil.copy(f'{DRIVE}/indexes/bm25_medcorp_tp.pkl', BM25)
        os.makedirs(FAISS, exist_ok=True)
        for f in ('faiss.index', 'chunks.pkl'):
            shutil.copy(f'{DRIVE}/indexes/faiss_medcorp_tp/{f}', f'{FAISS}/{f}')
        print(f'done in {time.time() - t:.0f}s')
    else:
        print('indexes already present locally')
else:
    # Fallback: rebuild from the corpus on the GPU (~25 min).
    print('no prebuilt indexes — rebuilding from the corpus')
    if not os.path.exists('data/corpora/medcorp_tp.jsonl'):
        # Symlink, so the DOWNLOAD stage is skipped: re-fetching 426K chunks from
        # HuggingFace when the corpus is already on Drive would be pure waste.
        os.symlink(corpus, 'data/corpora/medcorp_tp.jsonl')

    # Shards are checkpointed to disk, so a disconnect mid-embed loses at most one
    # shard -- just re-run this cell.
    print('embedding corpus on GPU (~10 min)...')
    !python scripts/build_medcorp.py --stage embed --name medcorp_tp
    print('assembling FAISS + BM25 indexes (~10 min)...')
    !python scripts/build_medcorp.py --stage assemble --name medcorp_tp

    # Cache to Drive so later sessions skip all of this.
    print('caching indexes to Drive...')
    shutil.copy(BM25, f'{DRIVE}/indexes/')
    os.makedirs(f'{DRIVE}/indexes/faiss_medcorp_tp', exist_ok=True)
    for f in ('faiss.index', 'chunks.pkl'):
        shutil.copy(f'{FAISS}/{f}', f'{DRIVE}/indexes/faiss_medcorp_tp/{f}')

for p in (BM25, f'{FAISS}/faiss.index', f'{FAISS}/chunks.pkl'):
    assert os.path.exists(p), f'missing after setup: {p}'
    print(f'  {p}  {os.path.getsize(p) / 1e6:.0f} MB')
print('indexes ready')


## 6. 🔴 SMOKE TEST — do not skip

**This is the gate. Nothing long runs until it passes.**

It checks the four things that can silently ruin a 22-hour run:

1. **Chat format** — the prompt uses Qwen's ChatML, not Llama's `[INST]`. Wrong markup degrades answers *without erroring*.
2. **Answers parse** — `extract_letter` finds a letter. If not, accuracy reads ~0% and looks like a bad model rather than a bad parser.
3. **Gates fire** — entropy and margin return real numbers, i.e. `logits_all` survived the CUDA build. If they are dead, P5 silently degenerates into retrieve-everything.
4. **Signals span the threshold** — if every query lands on one side of τ, the gate never fires. That is *expected* (it is the calibration-non-transfer finding) and is fixed offline in step 8 — but it must be caught **now**, not after the run.

In [ ]:
!python scripts/smoke_test_model.py \
  --model-config {spec['cfg']} \
  --dataset data/raw/mirage/benchmark.json \
  --bm25-index {BM25} --faiss-index {FAISS} \
  -n 20

## 6b. Calibrate the thresholds — BEFORE the long runs, not after

τ = 0.70/0.70 was fitted to the **3B's** signal distribution. The project's
calibration-non-transfer finding *predicts it will not transfer* to Qwen. If it does
not, and we only find out afterwards, P5 will have spent the night collapsed onto
P1 (retrieve everything) or P3 (retrieve nothing), measuring nothing.

So a **40-question** P5 run harvests Qwen's signal distribution (~20 min), the
thresholds are refitted offline in seconds, and only then do the 200-question runs
start.

**How the refit works — and why not just copy τ, or use the script's p75/p25 hint.**
The sweep script prints "suggested τ_H (entropy p75), τ_M (margin p25)". Those were
the *plan's starting points*, not what the 3B shipped. Measured on the real 3B log:

| | τ_H/τ_M | entropy | margin | probe | ensemble |
|---|---|---|---|---|---|
| shipped 3B | 0.70/0.70 | 52% | 54% | 40% | **50%** |
| p75/p25 hint | 0.919/0.635 | 25% | 25% | 40% | **24%** |

Using p75/p25 would hand Qwen **half** the 3B's retrieval budget, confounding model
scale with how often each model is allowed to retrieve — and the central claim
(*selective beats always-retrieve*) would no longer be comparable across models.

Instead each gate's τ is set so it **fires at the same rate it did on the 3B**. The
retrieval budget is held constant, scale is the only variable, and non-transfer
becomes a sharper claim: not "τ stopped working" but "τ had to move *this far* to
buy the same budget."

The 50 calibration questions are a **stride sample** (every 4th) of the same 200, so
they span every subject. Fitting on a prefix instead overshoots the retrieval budget
by ~6pp — and that error does not shrink with more prefix, because more prefix is
still the same subjects.

*Caveat for the write-up:* the calibration questions are a subset of the evaluation
set, so the operating point is not fitted on held-out data. The 3B was calibrated the
same way (replay over its own run), so the cross-model comparison is consistent —
but state it as a limitation.

In [ ]:
# ~12 min. Harvests gate signals only; this short run's accuracy is not reported.
#
# Calibrates on a STRIDE sample (every 4th of the 200), not the first 50. The
# evaluation set is 200 MMLU questions ordered by subject, so a prefix is one or two
# subjects rather than a sample of the benchmark. Measured on the 3B's own signals,
# a prefix overshoots the retrieval budget by ~6pp and the error does not shrink
# with N; the stride sample lands within ~1pp. See scripts/make_calibration_set.py.
TAG = f'qwen{MODEL}'
CAL_LOG = f'{DRIVE}/logs/p5_{TAG}_calib.jsonl'
CAL_SET = 'data/raw/mirage/calib50.json'
CAL_N = 50

import os, subprocess
if not os.path.exists(CAL_SET):
    subprocess.run(['python', 'scripts/make_calibration_set.py',
                    '--output', CAL_SET, '-n', str(CAL_N)], check=True)

done = sum(1 for _ in open(CAL_LOG)) if os.path.exists(CAL_LOG) else 0
if done < CAL_N:
    cmd = ['python', 'scripts/run_experiment.py',
           '--model-config', spec['cfg'],
           '--policy', 'configs/policies/p5_gated_entropy.yaml',
           '--experiment', 'configs/experiments/mirage_medcorp.yaml',
           '--dataset', CAL_SET,
           '--retrieval-mode', 'hybrid',
           '--bm25-index', BM25, '--faiss-index', FAISS,
           '--max-questions', str(CAL_N),
           '--output', CAL_LOG] + GGUF_OVERRIDE
    subprocess.run(cmd, check=True)
else:
    print(f'calibration log already has {done} records')

!python scripts/run_threshold_sweep.py --logs {CAL_LOG}:QWEN_CAL


In [ ]:
# Calibrate Qwen by matching the 3B's RETRIEVAL BUDGET, not by copying its tau.
#
# Measured on the 3B at its shipped tau=0.70/0.70 (results/raw_logs/p5_medcorp_mcq.jsonl,
# n=200). Hard-coded because results/raw_logs/ is gitignored and so is not present
# in the Colab checkout.
TARGET_ENTROPY_RATE = 0.52   # fraction of queries where the entropy gate votes retrieve
TARGET_MARGIN_RATE  = 0.54   # ditto, margin gate
TARGET_ENSEMBLE     = 0.50   # resulting 3-gate majority rate on the 3B

import sys, shutil, statistics as st, re
sys.path.insert(0, 'scripts')
from run_threshold_sweep import load_signals, _pct, ensemble_rate, PROBE_THRESHOLD

_, sigs = load_signals(f'{CAL_LOG}:QWEN_CAL')
assert sigs, 'no gate signals in the calibration log — the gates did not fire at all'

ent = [s['entropy'] for s in sigs]
mar = [s['margin'] for s in sigs]
prb = [s['hallucination_probe'] for s in sigs]

for nm, xs in (('entropy', ent), ('margin', mar), ('probe', prb)):
    print(f'{nm:8s} min={min(xs):.3f} p25={_pct(xs,0.25):.3f} '
          f'median={st.median(xs):.3f} p75={_pct(xs,0.75):.3f} max={max(xs):.3f}')

# entropy votes retrieve when signal > tau_H, so to fire on the top 52% of queries
# tau_H sits at the 48th percentile. margin votes when signal < tau_M, so it sits
# at the 54th percentile directly.
TAU_H = round(_pct(ent, 1.0 - TARGET_ENTROPY_RATE), 3)
TAU_M = round(_pct(mar, TARGET_MARGIN_RATE), 3)

old = ensemble_rate(sigs, 0.70, 0.70)      # what the 3B's tau does on Qwen
new = ensemble_rate(sigs, TAU_H, TAU_M)    # budget-matched operating point

print(f'\n{"gate":22s} {"3B tau .70/.70":>15s} {"matched " + str(TAU_H) + "/" + str(TAU_M):>20s} {"3B actual":>10s}')
tgt = {'entropy': TARGET_ENTROPY_RATE, 'margin': TARGET_MARGIN_RATE,
       'hallucination_probe': 0.40, 'ensemble': TARGET_ENSEMBLE}
for g in ('entropy', 'margin', 'hallucination_probe', 'ensemble'):
    print(f'{g:22s} {old[g]:>14.0%} {new[g]:>20.0%} {tgt[g]:>10.0%}')

# --- THE FINDING -----------------------------------------------------------
# Non-transfer is now measured as a DISTANCE: how far tau must move to buy the
# same retrieval budget on a different model.
print(f'\ntau shift needed to hold the budget constant: '
      f'entropy 0.70 -> {TAU_H} ({TAU_H - 0.70:+.3f}), '
      f'margin 0.70 -> {TAU_M} ({TAU_M - 0.70:+.3f})')
if old['ensemble'] in (0.0, 1.0):
    print('NON-TRANSFER: CONFIRMED — the 3B operating point is fully degenerate on Qwen.')
else:
    print(f'NON-TRANSFER: the 3B tau still fires here, but at {old["ensemble"]:.0%} '
          f'vs the intended {TARGET_ENSEMBLE:.0%}.')

# --- failure modes to catch BEFORE committing the night --------------------
# The probe has no threshold on MCQ (letter_match: two drafts disagree -> retrieve),
# so it cannot be recalibrated. If a stronger model simply agrees with itself more
# often, the probe goes quiet and majority-2-of-3 reduces to "entropy AND margin" —
# two gates that already agree 82% of the time. That is a real result about gate
# ensembles at scale, but it has to be seen now, not discovered in the results.
if new['hallucination_probe'] < 0.05:
    print(f'\n[WARN] probe votes retrieve on only {new["hallucination_probe"]:.0%} of queries '
          f'(3B: 40%) — effectively silent.\n'
          f'       The ensemble is now entropy+margin alone. Not a bug: 7B self-agrees\n'
          f'       more than 3B. Report it as a finding.')
elif new['hallucination_probe'] > 0.95:
    print(f'\n[WARN] probe votes retrieve on {new["hallucination_probe"]:.0%} of queries — '
          f'always-on, adds no discrimination.')

if not 0.15 < new['ensemble'] < 0.85:
    print(f'\n[WARN] ensemble retrieval {new["ensemble"]:.0%} is near-degenerate; P5 will '
          f'behave much like {"P3" if new["ensemble"] < 0.5 else "P1"}.')

# The open-ended probe threshold (base.yaml hallucination_probe.f1_threshold = 0.7)
# is deliberately NOT refit: holding the probe's definition constant is what keeps
# it comparable across models. Note it in the limitations section.
print(f'\nprobe f1_threshold held at {PROBE_THRESHOLD} across models — by design.')

P5_POLICY = 'p5_gated_qwen'
src = 'configs/policies/p5_gated_entropy.yaml'
dst = f'configs/policies/{P5_POLICY}.yaml'
text = open(src, encoding='utf-8').read()
text = re.sub(r'entropy_threshold: [\d.]+', f'entropy_threshold: {TAU_H}', text)
text = re.sub(r'margin_threshold: [\d.]+',  f'margin_threshold: {TAU_M}',  text)
text += (f"\n# Refitted for {spec['cfg']} on {len(sigs)} calibration questions by matching\n"
         f"# the 3B's per-gate retrieval budget (entropy {TARGET_ENTROPY_RATE:.0%}, "
         f"margin {TARGET_MARGIN_RATE:.0%}), NOT by copying its tau.\n"
         f"# The 3B tau 0.70/0.70 would have given {old['ensemble']:.0%} ensemble retrieval\n"
         f"# here; the matched point gives {new['ensemble']:.0%} (3B: {TARGET_ENSEMBLE:.0%}).\n")
open(dst, 'w', encoding='utf-8').write(text)
print(f'\nwrote {dst}  ->  P5_POLICY = {P5_POLICY!r}')

# The repo checkout is wiped when the VM recycles; keep the fitted policy on Drive.
shutil.copy(dst, f'{DRIVE}/logs/{P5_POLICY}.yaml')


## 7. Run the policies

All four policies, MCQ and open-ended. Logs are written **straight to Drive**, and every run skips question-ids already present — so if Colab disconnects, just re-run this cell and it picks up where it stopped.

Start with P3 and P5: they are the informative pair (P3 sets the new ceiling; P5 tests whether *selective beats always* survives the scale change). P1 and P4 complete the comparison.

In [ ]:
import subprocess, os

MCQ  = dict(dataset='data/raw/mirage/benchmark.json',
            experiment='configs/experiments/mirage_medcorp.yaml')
OPEN = dict(dataset='data/raw/openqa/pubmedqa_labeled.jsonl',
            experiment='configs/experiments/pubmedqa_open.yaml')

assert 'P5_POLICY' in dir(), 'Run the 6b calibration cells first — otherwise P5 ' \
                             'would silently use the 3B thresholds.'

# Order is deliberate: the two cheap P3 runs finish first, so a disconnect at any
# point still leaves a usable result (the new closed-book ceiling) rather than
# nothing. P5 is the expensive pair and goes last.
RUNS = [
    ('p3_closed_book', 'none',   'mcq',  MCQ),
    ('p3_closed_book', 'none',   'open', OPEN),
    (P5_POLICY,        'hybrid', 'mcq',  MCQ),
    (P5_POLICY,        'hybrid', 'open', OPEN),
]

for policy, mode, kind, ds in RUNS:
    out = f'{DRIVE}/logs/{policy.split("_")[0]}_{TAG}_{kind}.jsonl'
    done = sum(1 for _ in open(out)) if os.path.exists(out) else 0
    if done >= 200:
        print(f'[skip] {policy} {kind}: already {done}/200')
        continue
    print(f'\n=== {policy} {kind} ({done}/200 done) ===', flush=True)
    cmd = [
        'python', 'scripts/run_experiment.py',
        '--model-config', spec['cfg'],
        '--policy', f'configs/policies/{policy}.yaml',
        '--experiment', ds['experiment'],
        '--dataset', ds['dataset'],
        '--retrieval-mode', mode,
        '--output', out,
    ] + GGUF_OVERRIDE
    if mode != 'none':
        cmd += ['--bm25-index', BM25, '--faiss-index', FAISS]
    subprocess.run(cmd, check=False)   # keep going if one policy dies

print('\nAll runs attempted. Re-run this cell after any disconnect to resume.')


## 8. Confirm the calibration on the full run

The thresholds were fitted in step 6b on 40 questions. This replays the **full 200**
so the write-up can report the complete signal distribution and confirm the subset
was representative. No model calls — seconds.

In [ ]:
# Post-hoc: the full 200-question signal distribution, for the write-up's
# calibration table. The thresholds were already fitted in step 6b; this confirms
# the 40-question subset was representative.
!python scripts/run_threshold_sweep.py \
  --logs {DRIVE}/logs/p5_{TAG}_mcq.jsonl:QWEN_MCQ \
         {DRIVE}/logs/p5_{TAG}_open.jsonl:QWEN_OPEN


## 9. Copy the logs back into the repo

Download `MyDrive/medrag/logs/*.jsonl` and drop them into `results/raw_logs/` on your machine. Then, locally:

```bash
python scripts/generate_tables.py     # numbers.tex picks up the new models
python scripts/generate_figures.py    # fig_scaling.png
python -m pytest tests/               # drift guard confirms text == logs
```

The three questions the curve answers — **each of which is a result whichever way it lands**:

1. Does **selective beats always-retrieve** still hold at 7B and 14B? (generalises the central claim beyond one model)
2. Does **calibration non-transfer** hold prospectively? (τ=0.70 should fail on Qwen)
3. **Where does the curve sit relative to the safety bars, and is it flattening?** If 14B clears the 70% low bar, that is the scale at which low-risk deployment becomes viable. If it does not, the model-capability-ceiling finding is *strengthened*.